# Introduction

Building neural networks is a complex endeavor with many parameters to tweak prior to achieving the final version of a model. On top of this, the two most widely used numerical platforms for deep learning and neural network machine learning models, TensorFlow and Theano, are too complex to allow for rapid prototyping. The Keras Deep Learning library for Python helps bridge the gap between prototyping speed and the utilization of the advanced numerical platforms for deep learning.

## Keras
![Keras Image](https://s3.amazonaws.com/keras.io/img/keras-logo-2018-large-1200.png)

Keras is a high-level neural networks API, written in Python and capable of running on top of TensorFlow, CNTK, or Theano. It was developed with a focus on enabling fast experimentation. Being able to go from idea to result with the least possible delay is key to doing good research.

Use Keras if you need a deep learning library that:

* Allows for easy and fast prototyping (through user friendliness, modularity, and extensibility).
* Supports both convolutional networks and recurrent networks, as well as combinations of the two.
* Runs seamlessly on CPU and GPU.

# Getting Started

## Problem Definition

In this problem, we will be using the famous IRIS Flower dataset.

This dataset is well studied and is a good problem for practicing on neural networks because all of the 4 input variables are numeric and have the same scale in centimeters. Each instance describes the properties of an observed flower measurements and the output variable is specific iris species.

This is a multi-class classification problem, meaning that there are more than two classes to be predicted, in fact there are three flower species. This is an important type of problem on which to practice with neural networks because the three class values require specialized handling.

The iris flower dataset is a well studied problem and a such we can expect to achieve an model accuracy in the range of 95% to 97%. This provides a good target to aim for when developing our models.

You can [download](http://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data) the iris flowers dataset from the UCI Machine Learning repository and place it in your current working directory with the filename iris.csv

However, in our case - we have imported the dataset from kaggle in sqlite3 format.

## Import libraries and Functions

In [34]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

import sqlite3

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler

import tensorflow as tf

> Tensorflow version

In [35]:
tf.__version__

'1.13.1'

> Intialization Seed

This is important to ensure that the results we achieve from this model can be achieved again precisely. It ensures that the stochastic process of training a neural network model can be reproduced.

In [36]:
# fix random seed for reproducibility
seed = 42
np.random.seed(seed)

## Load the dataset

Here, we will loading data from a a sqlite database using pandas. Alternatively, we can load the data from csv as well. Also, from sklearn.datasets as well

In [48]:
connection = sqlite3.connect('../input/database.sqlite')
data = pd.read_sql_query(''' SELECT * FROM IRIS ''', connection)
print("Shape of data: {}".format(data.shape))

Shape of data: (150, 6)


Printing the first 3 rows from the database will give a higher level understanding of "what's in the data actually"

In [38]:
data.head(3)

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa


Checking if the dataset contains null/na values or not.

In [39]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
Id               150 non-null int64
SepalLengthCm    150 non-null float64
SepalWidthCm     150 non-null float64
PetalLengthCm    150 non-null float64
PetalWidthCm     150 non-null float64
Species          150 non-null object
dtypes: float64(4), int64(1), object(1)
memory usage: 7.1+ KB


**Observation** - 
1. There are no null values in the dataset.
2. Total number of observations are 150.
3. All the features except the output feature i.e. Species are of float dtype.

In [40]:
Y = data['Species']
X = data.drop(['Id', 'Species'], axis=1)
print("Shape of Input  features: {}".format(X.shape))
print("Shape of Output features: {}".format(Y.shape))

Shape of Input  features: (150, 4)
Shape of Output features: (150,)


## Encoding the Output/Response Variable

In [41]:
Y.value_counts()

Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: Species, dtype: int64

In [42]:
lbl_clf = LabelEncoder()
Y_encoded = lbl_clf.fit_transform(Y)

#Keras requires your output feature to be one-hot encoded values.
Y_final = tf.keras.utils.to_categorical(Y_encoded)

print("Therefore, our final shape of output feature will be {}".format(Y_final.shape))

Therefore, our final shape of output feature will be (150, 3)


## Splitting the dataset in 75-25 ratio

In [43]:
x_train, x_test, y_train, y_test = train_test_split(X, Y_final, test_size=0.25, random_state=seed, stratify=Y_encoded, shuffle=True)

print("Training Input shape\t: {}".format(x_train.shape))
print("Testing Input shape\t: {}".format(x_test.shape))
print("Training Output shape\t: {}".format(y_train.shape))
print("Testing Output shape\t: {}".format(y_test.shape))

Training Input shape	: (112, 4)
Testing Input shape	: (38, 4)
Training Output shape	: (112, 3)
Testing Output shape	: (38, 3)


## Standardizing the dataset

In [44]:
std_clf = StandardScaler()
x_train_new = std_clf.fit_transform(x_train)
x_test_new = std_clf.transform(x_test)

In [45]:
model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Dense(10, input_dim=4, activation=tf.nn.relu, kernel_initializer='he_normal', 
                                kernel_regularizer=tf.keras.regularizers.l2(0.01)))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.Dropout(0.3))
model.add(tf.keras.layers.Dense(7, activation=tf.nn.relu, kernel_initializer='he_normal', 
                                kernel_regularizer=tf.keras.regularizers.l1_l2(l1=0.001, l2=0.001)))
model.add(tf.keras.layers.BatchNormalization())
model.add(tf.keras.layers.Dropout(0.3))
model.add(tf.keras.layers.Dense(5, activation=tf.nn.relu, kernel_initializer='he_normal', 
                                kernel_regularizer=tf.keras.regularizers.l1_l2(l1=0.001, l2=0.001)))
model.add(tf.keras.layers.Dense(3, activation=tf.nn.softmax))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

iris_model = model.fit(x_train_new, y_train, epochs=700, batch_size=7)

Epoch 1/500
112/112 [==============================] - 1s 5ms/sample - loss: 1.3863 - acc: 0.4911
Epoch 2/500
112/112 [==============================] - 0s 813us/sample - loss: 1.2275 - acc: 0.5268
Epoch 3/500
112/112 [==============================] - 0s 826us/sample - loss: 1.2243 - acc: 0.4464
Epoch 4/500
112/112 [==============================] - 0s 835us/sample - loss: 1.2161 - acc: 0.5268
Epoch 5/500
112/112 [==============================] - 0s 840us/sample - loss: 1.0740 - acc: 0.5357
Epoch 6/500
112/112 [==============================] - 0s 855us/sample - loss: 1.1224 - acc: 0.5982
Epoch 7/500
112/112 [==============================] - 0s 828us/sample - loss: 1.0368 - acc: 0.5893
Epoch 8/500
112/112 [==============================] - 0s 773us/sample - loss: 1.0901 - acc: 0.6161
Epoch 9/500
112/112 [==============================] - 0s 794us/sample - loss: 1.0330 - acc: 0.5804
Epoch 10/500
112/112 [==============================] - 0s 849us/sample - loss: 0.9300 - acc: 0.6339
E

112/112 [==============================] - 0s 840us/sample - loss: 0.5046 - acc: 0.8482
Epoch 83/500
112/112 [==============================] - 0s 778us/sample - loss: 0.5384 - acc: 0.8750
Epoch 84/500
112/112 [==============================] - 0s 762us/sample - loss: 0.4744 - acc: 0.8839
Epoch 85/500
112/112 [==============================] - 0s 802us/sample - loss: 0.5760 - acc: 0.8125
Epoch 86/500
112/112 [==============================] - 0s 827us/sample - loss: 0.5242 - acc: 0.8304
Epoch 87/500
112/112 [==============================] - 0s 822us/sample - loss: 0.6349 - acc: 0.8036
Epoch 88/500
112/112 [==============================] - 0s 760us/sample - loss: 0.4937 - acc: 0.8839
Epoch 89/500
112/112 [==============================] - 0s 773us/sample - loss: 0.4649 - acc: 0.8839
Epoch 90/500
112/112 [==============================] - 0s 810us/sample - loss: 0.5260 - acc: 0.8125
Epoch 91/500
112/112 [==============================] - 0s 825us/sample - loss: 0.6760 - acc: 0.7321
Epo

112/112 [==============================] - 0s 783us/sample - loss: 0.4347 - acc: 0.8571
Epoch 163/500
112/112 [==============================] - 0s 785us/sample - loss: 0.4945 - acc: 0.8393
Epoch 164/500
112/112 [==============================] - 0s 772us/sample - loss: 0.4576 - acc: 0.8571
Epoch 165/500
112/112 [==============================] - 0s 798us/sample - loss: 0.3551 - acc: 0.9107
Epoch 166/500
112/112 [==============================] - 0s 833us/sample - loss: 0.4803 - acc: 0.8482
Epoch 167/500
112/112 [==============================] - 0s 843us/sample - loss: 0.4167 - acc: 0.8482
Epoch 168/500
112/112 [==============================] - 0s 837us/sample - loss: 0.3791 - acc: 0.8839
Epoch 169/500
112/112 [==============================] - 0s 756us/sample - loss: 0.3427 - acc: 0.9196
Epoch 170/500
112/112 [==============================] - 0s 809us/sample - loss: 0.4179 - acc: 0.8661
Epoch 171/500
112/112 [==============================] - 0s 795us/sample - loss: 0.3731 - acc: 0

112/112 [==============================] - 0s 790us/sample - loss: 0.3527 - acc: 0.8839
Epoch 243/500
112/112 [==============================] - 0s 801us/sample - loss: 0.3363 - acc: 0.8750
Epoch 244/500
112/112 [==============================] - 0s 728us/sample - loss: 0.4007 - acc: 0.8839
Epoch 245/500
112/112 [==============================] - 0s 773us/sample - loss: 0.3011 - acc: 0.9196
Epoch 246/500
112/112 [==============================] - 0s 772us/sample - loss: 0.4643 - acc: 0.8571
Epoch 247/500
112/112 [==============================] - 0s 815us/sample - loss: 0.4017 - acc: 0.9018
Epoch 248/500
112/112 [==============================] - 0s 812us/sample - loss: 0.6057 - acc: 0.7857
Epoch 249/500
112/112 [==============================] - 0s 811us/sample - loss: 0.4779 - acc: 0.8482
Epoch 250/500
112/112 [==============================] - 0s 759us/sample - loss: 0.4181 - acc: 0.8839
Epoch 251/500
112/112 [==============================] - 0s 810us/sample - loss: 0.3539 - acc: 0

112/112 [==============================] - 0s 828us/sample - loss: 0.3730 - acc: 0.8750
Epoch 323/500
112/112 [==============================] - 0s 781us/sample - loss: 0.3488 - acc: 0.8571
Epoch 324/500
112/112 [==============================] - 0s 789us/sample - loss: 0.2785 - acc: 0.9196
Epoch 325/500
112/112 [==============================] - 0s 782us/sample - loss: 0.2827 - acc: 0.9107
Epoch 326/500
112/112 [==============================] - 0s 807us/sample - loss: 0.2967 - acc: 0.9018
Epoch 327/500
112/112 [==============================] - 0s 782us/sample - loss: 0.3640 - acc: 0.8750
Epoch 328/500
112/112 [==============================] - 0s 791us/sample - loss: 0.3543 - acc: 0.9018
Epoch 329/500
112/112 [==============================] - 0s 779us/sample - loss: 0.3516 - acc: 0.8839
Epoch 330/500
112/112 [==============================] - 0s 780us/sample - loss: 0.2642 - acc: 0.8750
Epoch 331/500
112/112 [==============================] - 0s 857us/sample - loss: 0.3338 - acc: 0

112/112 [==============================] - 0s 815us/sample - loss: 0.2880 - acc: 0.9286
Epoch 403/500
112/112 [==============================] - 0s 749us/sample - loss: 0.3024 - acc: 0.8929
Epoch 404/500
112/112 [==============================] - 0s 759us/sample - loss: 0.2998 - acc: 0.9196
Epoch 405/500
112/112 [==============================] - 0s 792us/sample - loss: 0.2973 - acc: 0.9018
Epoch 406/500
112/112 [==============================] - 0s 793us/sample - loss: 0.1977 - acc: 0.9554
Epoch 407/500
112/112 [==============================] - 0s 770us/sample - loss: 0.4486 - acc: 0.8839
Epoch 408/500
112/112 [==============================] - 0s 814us/sample - loss: 0.2875 - acc: 0.8929
Epoch 409/500
112/112 [==============================] - 0s 781us/sample - loss: 0.3375 - acc: 0.8839
Epoch 410/500
112/112 [==============================] - 0s 791us/sample - loss: 0.2865 - acc: 0.9018
Epoch 411/500
112/112 [==============================] - 0s 835us/sample - loss: 0.3545 - acc: 0

112/112 [==============================] - 0s 771us/sample - loss: 0.2674 - acc: 0.8929
Epoch 483/500
112/112 [==============================] - 0s 786us/sample - loss: 0.4308 - acc: 0.8929
Epoch 484/500
112/112 [==============================] - 0s 773us/sample - loss: 0.2887 - acc: 0.9286
Epoch 485/500
112/112 [==============================] - 0s 785us/sample - loss: 0.3168 - acc: 0.9196
Epoch 486/500
112/112 [==============================] - 0s 779us/sample - loss: 0.2723 - acc: 0.8839
Epoch 487/500
112/112 [==============================] - 0s 796us/sample - loss: 0.3500 - acc: 0.8750
Epoch 488/500
112/112 [==============================] - 0s 834us/sample - loss: 0.3971 - acc: 0.8571
Epoch 489/500
112/112 [==============================] - 0s 817us/sample - loss: 0.3731 - acc: 0.8839
Epoch 490/500
112/112 [==============================] - 0s 751us/sample - loss: 0.1953 - acc: 0.9375
Epoch 491/500
112/112 [==============================] - 0s 749us/sample - loss: 0.3261 - acc: 0

In [47]:
model.evaluate(x_test_new, y_test)

38/38 [==============================] - 0s 188us/sample - loss: 0.1379 - acc: 1.0000


[0.1379205886470644, 1.0]